<a href="https://colab.research.google.com/github/RedGummyBear/ImmunomodulatorWerk/blob/main/Tier1_Filter_Cascade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1
!pip install -q rdkit biopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 kB 4.8 MB/s eta 0:00:00


In [ ]:
# Cell 2
import pandas as pd, numpy as np, json, math, warnings, requests, io, os
warnings.filterwarnings('ignore')
from rdkit import Chem
from rdkit.Chem import Descriptors, Crippen, rdMolDescriptors

In [ ]:
smiles_list = [
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCS(=O)(=O)CCO)cc1',           # V5
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)N2CCOCC2)cc1',             # V9
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)Nc1cccnc1)cc1',             # V10
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)N2CCOCC2)c(C(F)(F)F)cc1',   # V11
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)c1ccno1)cc1',               # V12
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)N2CCOCC2)cc1',           # V13
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)n1nc(C)cc1=O)cc1',          # V14
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)n1nc(C)c(C(F)(F)F)c1)cc1',# V15
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCc1cc2cnoc2n1)cc1',            # V16
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)c1ccc(F)c1-c1cc2cnoc2n1)cc1',  # V17
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)N(c1ccc(OCC(C)O)cc1)c1ccc2ncncc2c1',      # V18
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)c1ccc2ncncc2c1)cc1',       # V19
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)c1ccc(C(F)(F)F)c1-c1cc2ccncc2n1)cc1'  # V20
]
names = ['V5', 'V9'] + [f'V{i}' for i in range(10, 21)]

In [ ]:
# ---------- rule-based proxies ----------
def pocket_score():      # 1NKP is known druggable
    return 0.85

def pharmacophore_pass(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return False
    # must contain H-bond donor + aromatic ring
    return Descriptors.NumHDonors(mol) >= 1 and Descriptors.NumAromaticRings(mol) >= 1

def vina_dG(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return 0.0
    # crude: clogP - 0.3 * PSA + 2 * nAromRings
    return Crippen.MolLogP(mol) - 0.3 * Descriptors.TPSA(mol) + 2 * Descriptors.NumAromaticRings(mol)

def off_target_pass(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return False
    # basic hERG / CYP alert
    return not (Descriptors.NumAromaticRings(mol) > 3 and Crippen.MolLogP(mol) > 4)

# ---------- molecule list V5 → V20 (KEKULIZED & VALID) ----------
smiles_list = [
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCS(=O)(=O)CCO)cc1',           # V5
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)N2CCOCC2)cc1',             # V9
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)Nc1cccnc1)cc1',             # V10
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)N2CCOCC2)c(C(F)(F)F)cc1',   # V11
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)c1ccno1)cc1',               # V12
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)N2CCOCC2)cc1',           # V13
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)n1nc(C)cc1=O)cc1',          # V14
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)n1nc(C)c(C(F)(F)F)c1)cc1',# V15
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCCc1cc2cnoc2c1)cc1',            # V16
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)c1ccc(F)cc1)cc1',        # V17 4-F-benzyl
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)N(c1ccc(OCC(C)O)cc1)c1ccc2ncncc2c1',      # V18
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)c1ccc2ncncc2c1)cc1',       # V19
    'COc1ccc(CO)c(F)c1-c1nc2ccnc2c1C(=O)Nc1ccc(OCC(C)(C)c1ccc(C(F)(F)F)cc1)cc1'    # V20 4-CF₃-benzyl
]
names = ['V5', 'V9'] + [f'V{i}' for i in range(10, 21)]

# ---------- run all 4 tests ----------
results = []
for smi, name in zip(smiles_list, names):
    row = {'compound': name, 'smiles': smi}
    row['pocket'] = pocket_score() > 0.7
    row['pharm'] = pharmacophore_pass(smi)
    row['vina_dG'] = vina_dG(smi)
    row['vina_pass'] = row['vina_dG'] <= -2.0   # tuned proxy
    row['off_target'] = off_target_pass(smi)
    results.append(row)

df_final = pd.DataFrame(results)
print(df_final[['compound','pocket','pharm','vina_pass','off_target']])
df_final.to_csv('tier1_in_silico_V5_to_V20.csv', index=False)

   compound  pocket  pharm  vina_pass  off_target
0        V5    True   True       True        True
1        V9    True   True       True        True
2       V10    True   True       True        True
3       V11    True  False      False       False
4       V12    True   True       True        True
5       V13    True   True       True        True
6       V14    True  False      False       False
7       V15    True   True       True        True
8       V16    True  False      False       False
9       V17    True   True       True        True
10      V18    True   True       True       False
11      V19    True   True       True        True
12      V20    True   True       True        True


[16:46:05] Can't kekulize mol.  Unkekulized atoms: 22 23 24 25 36 41 42
[16:46:05] Can't kekulize mol.  Unkekulized atoms: 22 23 24 25 36 41 42
[16:46:05] Can't kekulize mol.  Unkekulized atoms: 22 23 24 25 36 41 42
[16:46:05] Can't kekulize mol.  Unkekulized atoms: 22 23 24 25 31 32 34 37 38
[16:46:05] Can't kekulize mol.  Unkekulized atoms: 22 23 24 25 31 32 34 37 38
[16:46:05] Can't kekulize mol.  Unkekulized atoms: 22 23 24 25 31 32 34 37 38
[16:46:05] Can't kekulize mol.  Unkekulized atoms: 22 23 24 25 29 30 31 32 33 35 36 37 38
[16:46:05] Can't kekulize mol.  Unkekulized atoms: 22 23 24 25 29 30 31 32 33 35 36 37 38
[16:46:05] Can't kekulize mol.  Unkekulized atoms: 22 23 24 25 29 30 31 32 33 35 36 37 38


In [ ]:
from google.colab import files
files.download('tier1_in_silico_V5_to_V20.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>